In [ ]:
simport random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from pathlib import Path
import sys

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

SRC_PATH = PROJECT_PATH / "src"

PROCESSED_PATH = PROJECT_PATH / "datasets" / "processed"

sys.path.append(str(SRC_PATH))

In [ ]:
data = np.load(
    PROCESSED_PATH / "uah_dataset.npz"
)

X = data["X"]
y = data["y"]
groups = data["groups"]

In [ ]:
print("=" * 60)
print("Dataset Information")
print("=" * 60)

print("X :", X.shape)
print("y :", y.shape)
print("groups :", groups.shape)

print()

print("Class Distribution")

classes, counts = np.unique(y, return_counts=True)

for c, n in zip(classes, counts):
    print(f"Class {c}: {n}")

Dataset Information
X : (30676, 120, 13)
y : (30676,)
groups : (30676,)

Class Distribution
Class 0: 12991
Class 1: 9846
Class 2: 7839


# ==========================================================
# 3. TRAIN / TEST SPLIT (TRIP-BASED)
# ==========================================================

In [ ]:
from sklearn.model_selection import train_test_split

# Her trip için benzersiz grup numaraları
unique_groups = np.unique(groups)

print("Trip Count:", len(unique_groups))

# NB43 ile aynı split
train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=0.20,
    random_state=42,
)

print("Train Trips:", len(train_groups))
print("Test Trips :", len(test_groups))

Trip Count: 40
Train Trips: 32
Test Trips : 8


In [ ]:
train_mask = np.isin(groups, train_groups)
test_mask = np.isin(groups, test_groups)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

Train : (24115, 120, 13)
Test  : (6561, 120, 13)


# ==========================================================
# 4. CREATE DATALOADERS
# ==========================================================

In [ ]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import create_dataloader

train_loader = create_dataloader(
    X_train,
    y_train,
    batch_size=32,
    shuffle=True,
)

test_loader = create_dataloader(
    X_test,
    y_test,
    batch_size=32,
    shuffle=False,
)

print("Train Loader:", len(train_loader))
print("Test Loader :", len(test_loader))

Train Loader: 754
Test Loader : 206


# ==========================================================
# 5. LSTM MODEL
# ==========================================================

In [ ]:
import importlib
import lstm_model

importlib.reload(lstm_model)

from lstm_model import LSTMClassifier

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
model = LSTMClassifier().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

print(model)

LSTMClassifier(
  (lstm): LSTM(13, 64, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=3, bias=True)
  )
)


In [ ]:
# ==========================================================
# CONFIGURATION
# ==========================================================

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 20
PATIENCE = 3

# ==========================================================
# 6. BASELINE TRAINING
# ==========================================================

In [ ]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import fit

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.8964 | Train Acc: 0.5427 | Val Loss: 1.2220 | Val Acc: 0.4662
Epoch 2/20 | Train Loss: 0.7974 | Train Acc: 0.6065 | Val Loss: 0.9672 | Val Acc: 0.4656
Epoch 3/20 | Train Loss: 0.7465 | Train Acc: 0.6368 | Val Loss: 1.0128 | Val Acc: 0.5769
Epoch 4/20 | Train Loss: 0.7111 | Train Acc: 0.6583 | Val Loss: 1.1036 | Val Acc: 0.4094
Epoch 5/20 | Train Loss: 0.6673 | Train Acc: 0.6877 | Val Loss: 0.9164 | Val Acc: 0.4783
Epoch 6/20 | Train Loss: 0.6384 | Train Acc: 0.7062 | Val Loss: 0.9496 | Val Acc: 0.4960

Early stopping at epoch 6
Best Validation Accuracy : 0.5769


# ==========================================================
# 7. EVALUATION
# ==========================================================

In [ ]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import evaluate_model

In [ ]:
labels, predictions = evaluate_model(
    model,
    test_loader,
    device,
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        labels,
        predictions,
        target_names=[
            "NORMAL",
            "AGGRESSIVE",
            "DROWSY",
        ]
    )
)

              precision    recall  f1-score   support

      NORMAL       0.37      0.98      0.53      1455
  AGGRESSIVE       0.91      0.37      0.53      3592
      DROWSY       0.84      0.67      0.75      1514

    accuracy                           0.58      6561
   macro avg       0.71      0.68      0.60      6561
weighted avg       0.78      0.58      0.58      6561



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, predictions)

print(cm)

[[1424   12   19]
 [2080 1341  171]
 [ 378  116 1020]]


# ==========================================================
# 8. EXPERIMENT 1 - STANDARD SCALER
# ==========================================================

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# (N,120,13) -> (N*120,13)
X_train_flat = X_train.reshape(-1, X_train.shape[-1])
X_test_flat = X_test.reshape(-1, X_test.shape[-1])

# Sadece train üzerinde öğren
scaler.fit(X_train_flat)

# Train ve test'i dönüştür
X_train_scaled = scaler.transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

# Eski şekline geri getir
X_train_scaled = X_train_scaled.reshape(X_train.shape)
X_test_scaled = X_test_scaled.reshape(X_test.shape)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(24115, 120, 13)
(6561, 120, 13)


In [ ]:
import importlib
import trainer

importlib.reload(trainer)

from trainer import create_dataloader

train_loader_scaled = create_dataloader(
    X_train_scaled,
    y_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader_scaled = create_dataloader(
    X_test_scaled,
    y_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train Loader:", len(train_loader_scaled))
print("Test Loader :", len(test_loader_scaled))

Train Loader: 754
Test Loader : 206


In [ ]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_scaled = LSTMClassifier().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model_scaled.parameters(),
    lr=LEARNING_RATE,
)

In [ ]:
history_scaled = fit(
    model=model_scaled,
    train_loader=train_loader_scaled,
    val_loader=test_loader_scaled,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.7241 | Train Acc: 0.6907 | Val Loss: 0.8527 | Val Acc: 0.6129
Epoch 2/20 | Train Loss: 0.5541 | Train Acc: 0.7789 | Val Loss: 1.0899 | Val Acc: 0.6295
Epoch 3/20 | Train Loss: 0.4586 | Train Acc: 0.8148 | Val Loss: 1.0578 | Val Acc: 0.6863
Epoch 4/20 | Train Loss: 0.3920 | Train Acc: 0.8471 | Val Loss: 1.5050 | Val Acc: 0.6225
Epoch 5/20 | Train Loss: 0.3485 | Train Acc: 0.8677 | Val Loss: 1.4368 | Val Acc: 0.6664
Epoch 6/20 | Train Loss: 0.3013 | Train Acc: 0.8877 | Val Loss: 1.5529 | Val Acc: 0.5761

Early stopping at epoch 6
Best Validation Accuracy : 0.6863


# ==========================================================
# 10. EXPERIMENT 2 - CLASS WEIGHTS
# ==========================================================


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

print(class_weights)

[0.69680421 1.28531073 1.27088274]


In [ ]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_weighted = LSTMClassifier().to(device)

weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(
    model_weighted.parameters(),
    lr=LEARNING_RATE,
)

In [ ]:
history_weighted = fit(
    model=model_weighted,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.9104 | Train Acc: 0.4922 | Val Loss: 0.9878 | Val Acc: 0.4268
Epoch 2/20 | Train Loss: 0.7890 | Train Acc: 0.5899 | Val Loss: 0.8611 | Val Acc: 0.6715
Epoch 3/20 | Train Loss: 0.7592 | Train Acc: 0.6128 | Val Loss: 0.8293 | Val Acc: 0.6307
Epoch 4/20 | Train Loss: 0.7260 | Train Acc: 0.6139 | Val Loss: 1.0124 | Val Acc: 0.6478
Epoch 5/20 | Train Loss: 0.6886 | Train Acc: 0.6273 | Val Loss: 0.7341 | Val Acc: 0.7916
Epoch 6/20 | Train Loss: 0.6551 | Train Acc: 0.6408 | Val Loss: 0.7453 | Val Acc: 0.7691
Epoch 7/20 | Train Loss: 0.6307 | Train Acc: 0.6495 | Val Loss: 1.1708 | Val Acc: 0.6984
Epoch 8/20 | Train Loss: 0.5984 | Train Acc: 0.6632 | Val Loss: 0.7257 | Val Acc: 0.7916

Early stopping at epoch 8
Best Validation Accuracy : 0.7916


In [ ]:
labels, predictions = evaluate_model(
    model_weighted,
    test_loader,
    device,
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        labels,
        predictions,
        target_names=[
            "NORMAL",
            "AGGRESSIVE",
            "DROWSY",
        ]
    )
)

              precision    recall  f1-score   support

      NORMAL       0.42      0.68      0.52      1455
  AGGRESSIVE       0.82      0.73      0.77      3592
      DROWSY       0.88      0.57      0.69      1514

    accuracy                           0.68      6561
   macro avg       0.70      0.66      0.66      6561
weighted avg       0.74      0.68      0.70      6561



# ==========================================================
# 12. EXPERIMENT 3 - LEARNING RATE = 1e-4
# ==========================================================


In [ ]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_lr = LSTMClassifier().to(device)

weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(
    model_lr.parameters(),
    lr=1e-4,
)

In [ ]:
import importlib
import trainer

importlib.reload(trainer)

<module 'trainer' from '/content/drive/MyDrive/UAH_Project/src/trainer.py'>

In [ ]:
history_lr = fit(
    model=model_lr,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.9964 | Train Acc: 0.4374 | Val Loss: 1.0567 | Val Acc: 0.4946
Epoch 2/20 | Train Loss: 0.8574 | Train Acc: 0.5373 | Val Loss: 0.9323 | Val Acc: 0.5889
Epoch 3/20 | Train Loss: 0.7987 | Train Acc: 0.5841 | Val Loss: 0.9023 | Val Acc: 0.5816
Epoch 4/20 | Train Loss: 0.7629 | Train Acc: 0.6128 | Val Loss: 0.8673 | Val Acc: 0.6292
Epoch 5/20 | Train Loss: 0.7343 | Train Acc: 0.6285 | Val Loss: 0.8123 | Val Acc: 0.6487
Epoch 6/20 | Train Loss: 0.7258 | Train Acc: 0.6304 | Val Loss: 0.8820 | Val Acc: 0.6403
Epoch 7/20 | Train Loss: 0.7087 | Train Acc: 0.6432 | Val Loss: 0.7995 | Val Acc: 0.6636
Epoch 8/20 | Train Loss: 0.6938 | Train Acc: 0.6471 | Val Loss: 0.9432 | Val Acc: 0.5818
Epoch 9/20 | Train Loss: 0.6839 | Train Acc: 0.6492 | Val Loss: 0.8470 | Val Acc: 0.6807
Epoch 10/20 | Train Loss: 0.6728 | Train Acc: 0.6505 | Val Loss: 0.8024 | Val Acc: 0.6946
Epoch 11/20 | Train Loss: 0.6570 | Train Acc: 0.6532 | Val Loss: 0.9207 | Val Acc: 0.6539
Epoch 12/20 | Train

In [ ]:
labels, predictions = evaluate_model(
    model_lr,
    test_loader,
    device,
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        labels,
        predictions,
        target_names=[
            "NORMAL",
            "AGGRESSIVE",
            "DROWSY",
        ]
    )
)

              precision    recall  f1-score   support

      NORMAL       0.56      0.87      0.68      1455
  AGGRESSIVE       0.94      0.68      0.79      3592
      DROWSY       0.74      0.82      0.77      1514

    accuracy                           0.75      6561
   macro avg       0.74      0.79      0.75      6561
weighted avg       0.81      0.75      0.76      6561



# ==========================================================
# 13. EXPERIMENT 4
# StandardScaler + ClassWeights + LR = 1e-4
# ==========================================================

In [ ]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_scaled_lr = LSTMClassifier().to(device)

weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.Adam(
    model_scaled_lr.parameters(),
    lr=1e-4,
)

In [ ]:
history_scaled_lr = fit(
    model=model_scaled_lr,
    train_loader=train_loader_scaled,
    val_loader=test_loader_scaled,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.9275 | Train Acc: 0.4398 | Val Loss: 0.8280 | Val Acc: 0.5995
Epoch 2/20 | Train Loss: 0.7153 | Train Acc: 0.6213 | Val Loss: 0.7971 | Val Acc: 0.6126
Epoch 3/20 | Train Loss: 0.6536 | Train Acc: 0.6633 | Val Loss: 0.6854 | Val Acc: 0.6278
Epoch 4/20 | Train Loss: 0.6104 | Train Acc: 0.6950 | Val Loss: 0.6812 | Val Acc: 0.6027
Epoch 5/20 | Train Loss: 0.5777 | Train Acc: 0.7282 | Val Loss: 0.7340 | Val Acc: 0.6519
Epoch 6/20 | Train Loss: 0.5437 | Train Acc: 0.7560 | Val Loss: 0.8840 | Val Acc: 0.6405
Epoch 7/20 | Train Loss: 0.5042 | Train Acc: 0.7859 | Val Loss: 1.0298 | Val Acc: 0.6312
Epoch 8/20 | Train Loss: 0.4671 | Train Acc: 0.8051 | Val Loss: 0.8823 | Val Acc: 0.7165
Epoch 9/20 | Train Loss: 0.4682 | Train Acc: 0.7917 | Val Loss: 1.0001 | Val Acc: 0.6696
Epoch 10/20 | Train Loss: 0.4323 | Train Acc: 0.8081 | Val Loss: 0.9945 | Val Acc: 0.6987
Epoch 11/20 | Train Loss: 0.4033 | Train Acc: 0.8252 | Val Loss: 1.0366 | Val Acc: 0.6898

Early stopping at 

In [ ]:
labels, predictions = evaluate_model(
    model_scaled_lr,
    test_loader_scaled,
    device,
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        labels,
        predictions,
        target_names=[
            "NORMAL",
            "AGGRESSIVE",
            "DROWSY",
        ],
    )
)

              precision    recall  f1-score   support

      NORMAL       0.46      0.84      0.59      1455
  AGGRESSIVE       0.93      0.60      0.73      3592
      DROWSY       0.84      0.87      0.85      1514

    accuracy                           0.72      6561
   macro avg       0.74      0.77      0.73      6561
weighted avg       0.81      0.72      0.73      6561



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    labels,
    predictions,
)

print(cm)

[[1228  129   98]
 [1278 2151  163]
 [ 171   21 1322]]


# ==========================================================
# 14. EXPERIMENT 5 - GRADIENT CLIPPING
# ==========================================================

In [ ]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_clip = LSTMClassifier().to(device)

weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(
    model_clip.parameters(),
    lr=1e-4,
)

In [ ]:
history_clip = fit(
    model=model_clip,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 1.0315 | Train Acc: 0.4145 | Val Loss: 1.0491 | Val Acc: 0.4167
Epoch 2/20 | Train Loss: 0.9182 | Train Acc: 0.4835 | Val Loss: 1.1147 | Val Acc: 0.4291
Epoch 3/20 | Train Loss: 0.8512 | Train Acc: 0.5409 | Val Loss: 0.9137 | Val Acc: 0.5684
Epoch 4/20 | Train Loss: 0.8190 | Train Acc: 0.5752 | Val Loss: 0.8265 | Val Acc: 0.6411
Epoch 5/20 | Train Loss: 0.7781 | Train Acc: 0.6049 | Val Loss: 0.8869 | Val Acc: 0.5745
Epoch 6/20 | Train Loss: 0.7565 | Train Acc: 0.6160 | Val Loss: 0.7921 | Val Acc: 0.6549
Epoch 7/20 | Train Loss: 0.7339 | Train Acc: 0.6308 | Val Loss: 0.8017 | Val Acc: 0.6651
Epoch 8/20 | Train Loss: 0.7192 | Train Acc: 0.6366 | Val Loss: 0.7457 | Val Acc: 0.6851
Epoch 9/20 | Train Loss: 0.7022 | Train Acc: 0.6393 | Val Loss: 0.9438 | Val Acc: 0.6255
Epoch 10/20 | Train Loss: 0.6865 | Train Acc: 0.6426 | Val Loss: 0.9484 | Val Acc: 0.6354
Epoch 11/20 | Train Loss: 0.6604 | Train Acc: 0.6515 | Val Loss: 0.7820 | Val Acc: 0.7104
Epoch 12/20 | Train

In [ ]:
labels, predictions = evaluate_model(
    model_clip,
    test_loader,
    device,
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        labels,
        predictions,
        target_names=[
            "NORMAL",
            "AGGRESSIVE",
            "DROWSY",
        ],
    )
)

              precision    recall  f1-score   support

      NORMAL       0.54      0.79      0.64      1455
  AGGRESSIVE       0.93      0.74      0.82      3592
      DROWSY       0.78      0.81      0.80      1514

    accuracy                           0.77      6561
   macro avg       0.75      0.78      0.75      6561
weighted avg       0.81      0.77      0.78      6561



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    labels,
    predictions,
)

print(cm)

[[1149  166  140]
 [ 742 2649  201]
 [ 252   35 1227]]


# ==========================================================
# 15. EXPERIMENT 6 - HIDDEN SIZE = 128
# ==========================================================

In [ ]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_hidden128 = LSTMClassifier().to(device)

weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.Adam(
    model_hidden128.parameters(),
    lr=1e-4,
)

In [ ]:
history_hidden128 = fit(
    model=model_hidden128,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.9336 | Train Acc: 0.4686 | Val Loss: 1.0942 | Val Acc: 0.5226
Epoch 2/20 | Train Loss: 0.7928 | Train Acc: 0.5868 | Val Loss: 1.0097 | Val Acc: 0.6244
Epoch 3/20 | Train Loss: 0.7411 | Train Acc: 0.6183 | Val Loss: 0.8874 | Val Acc: 0.6502
Epoch 4/20 | Train Loss: 0.7054 | Train Acc: 0.6341 | Val Loss: 1.0322 | Val Acc: 0.5831
Epoch 5/20 | Train Loss: 0.6690 | Train Acc: 0.6496 | Val Loss: 1.0759 | Val Acc: 0.5735
Epoch 6/20 | Train Loss: 0.6332 | Train Acc: 0.6726 | Val Loss: 0.9861 | Val Acc: 0.6144

Early stopping at epoch 6
Best Validation Accuracy : 0.6502


# ==========================================================
# 16. EXPERIMENT 7 - BIDIRECTIONAL LSTM
# ==========================================================

In [ ]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_bilstm = LSTMClassifier().to(device)

weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights,
)

optimizer = torch.optim.Adam(
    model_bilstm.parameters(),
    lr=1e-4,
)

In [ ]:
history_bilstm = fit(
    model=model_bilstm,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 1.0091 | Train Acc: 0.4452 | Val Loss: 1.0340 | Val Acc: 0.6025
Epoch 2/20 | Train Loss: 0.8677 | Train Acc: 0.5487 | Val Loss: 0.9594 | Val Acc: 0.6065
Epoch 3/20 | Train Loss: 0.7966 | Train Acc: 0.5914 | Val Loss: 1.0496 | Val Acc: 0.5630
Epoch 4/20 | Train Loss: 0.7462 | Train Acc: 0.6271 | Val Loss: 0.9002 | Val Acc: 0.6280
Epoch 5/20 | Train Loss: 0.7173 | Train Acc: 0.6352 | Val Loss: 0.8461 | Val Acc: 0.6569
Epoch 6/20 | Train Loss: 0.6973 | Train Acc: 0.6498 | Val Loss: 0.9281 | Val Acc: 0.6418
Epoch 7/20 | Train Loss: 0.6765 | Train Acc: 0.6557 | Val Loss: 1.0533 | Val Acc: 0.5615
Epoch 8/20 | Train Loss: 0.6615 | Train Acc: 0.6620 | Val Loss: 0.8716 | Val Acc: 0.6610
Epoch 9/20 | Train Loss: 0.6431 | Train Acc: 0.6734 | Val Loss: 0.8811 | Val Acc: 0.6510
Epoch 10/20 | Train Loss: 0.6299 | Train Acc: 0.6739 | Val Loss: 0.9687 | Val Acc: 0.6633
Epoch 11/20 | Train Loss: 0.6083 | Train Acc: 0.6846 | Val Loss: 0.8463 | Val Acc: 0.6906
Epoch 12/20 | Train

In [ ]:
labels, predictions = evaluate_model(
    model_bilstm,
    test_loader,
    device,
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        labels,
        predictions,
        target_names=[
            "NORMAL",
            "AGGRESSIVE",
            "DROWSY",
        ],
    )
)

              precision    recall  f1-score   support

      NORMAL       0.45      0.89      0.60      1455
  AGGRESSIVE       0.92      0.61      0.73      3592
      DROWSY       0.79      0.68      0.73      1514

    accuracy                           0.69      6561
   macro avg       0.72      0.73      0.69      6561
weighted avg       0.79      0.69      0.70      6561



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, predictions)

print(cm)

[[1298   55  102]
 [1226 2196  170]
 [ 330  147 1037]]


In [ ]:
import importlib
import lstm_model

importlib.reload(lstm_model)

<module 'lstm_model' from '/content/drive/MyDrive/UAH_Project/src/lstm_model.py'>

# ==========================================================
# 17. EXPERIMENT 8
# BEST MODEL + STANDARD SCALER
# ==========================================================

In [ ]:
import importlib
import lstm_model
import trainer

importlib.reload(lstm_model)
importlib.reload(trainer)

from lstm_model import LSTMClassifier
from trainer import fit

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_best_scaled = LSTMClassifier().to(device)

weights = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.Adam(
    model_best_scaled.parameters(),
    lr=1e-4,
)

In [ ]:
history_best_scaled = fit(
    model=model_best_scaled,
    train_loader=train_loader_scaled,
    val_loader=test_loader_scaled,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.9436 | Train Acc: 0.4605 | Val Loss: 0.8950 | Val Acc: 0.5661
Epoch 2/20 | Train Loss: 0.7366 | Train Acc: 0.5925 | Val Loss: 0.7603 | Val Acc: 0.5359
Epoch 3/20 | Train Loss: 0.6567 | Train Acc: 0.6889 | Val Loss: 0.7240 | Val Acc: 0.5638
Epoch 4/20 | Train Loss: 0.6172 | Train Acc: 0.7186 | Val Loss: 0.7860 | Val Acc: 0.5551

Early stopping at epoch 4
Best Validation Accuracy : 0.5661


In [ ]:
front_distance = X[:, :, 10]

print("Shape:", front_distance.shape)
print("Number of -1 values:", (front_distance == -1).sum())

print("Minimum:", front_distance.min())
print("Maximum:", front_distance.max())

Shape: (30676, 120)
Number of -1 values: 1962128
Minimum: -1.0
Maximum: 222.72


In [ ]:
relative_speed = X[:, :, 11]

print("Number of -1 values:", (relative_speed == -1).sum())
print("Minimum:", relative_speed.min())
print("Maximum:", relative_speed.max())

Number of -1 values: 1962128
Minimum: -1.0
Maximum: 13.0
